In [1]:
import json
import numpy as np
from solution import *   # solution.py must be in the same folder as the notebook
from fetch_data import load_all

np.set_printoptions(suppress=True, precision=2)

In [2]:
import importlib, solution
importlib.reload(solution)
from solution import *

In [3]:
data = load_all()
income_records = data["hh_income_state"]
poverty_records = data["hh_poverty_state"]

print(type(income_records), len(income_records), len(poverty_records))
print("Income fields: ", list(income_records[0].keys()))
print("Poverty fields:", list(poverty_records[0].keys()))

<class 'list'> 319 310
Income fields:  ['date', 'state', 'income_mean', 'income_median']
Poverty fields: ['date', 'state', 'poverty_absolute', 'poverty_hardcore', 'poverty_relative']


In [4]:
with open("hh_income_state.json") as f:
    income_records = json.load(f)["records"]
with open("hh_poverty_state.json") as f:
    poverty_records = json.load(f)["records"]

In [5]:
validate_records(income_records, ("state", "date"))
validate_records(poverty_records, ("state", "date"))

STATES, MEDIAN, MEAN, POVERTY = build_matrices(income_records, poverty_records)
print(STATES)
print(MEDIAN.shape, MEAN.shape, POVERTY.shape)
print(np.isnan(MEDIAN).sum(), np.isnan(MEAN).sum(), np.isnan(POVERTY).sum())

['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']
(16, 5) (16, 5) (16, 4)
0 0 0


In [6]:
import numpy as np

S = np.array(STATES)
years = np.array(SURVEY_YEARS)

# 4.1 mean and median of the 16 state medians, per survey year
mean_of_medians = MEDIAN.mean(axis=0)
median_of_medians = np.median(MEDIAN, axis=0)
print("4.1 Mean / median of the 16 state medians")
for y, a, b in zip(years, mean_of_medians, median_of_medians):
    print(f"  {y}: mean = {a:.2f}, median = {b:.2f}")

# 4.2 percentage growth 2016 -> 2024 for all states at once
growth = (MEDIAN[:, -1] - MEDIAN[:, 0]) / MEDIAN[:, 0] * 100
print("\n4.2 Growth in median income 2016-2024")
for s, g in zip(S, growth):
    print(f"  {s} 2016-2024: {g:.2f}%")
print("  Highest:", S[growth.argmax()], f"{growth.max():.2f}%")
print("  Lowest: ", S[growth.argmin()], f"{growth.min():.2f}%")

# 4.3 highest and lowest median income in each survey year
highest = S[MEDIAN.argmax(axis=0)]
lowest = S[MEDIAN.argmin(axis=0)]
print("\n4.3 Highest / lowest median income by year")
for y, hi, lo in zip(years, highest, lowest):
    print(f"  {y}: highest = {hi}, lowest = {lo}")

# 4.4 relative income index: one broadcasting expression
rel_index = MEDIAN / np.median(MEDIAN, axis=0, keepdims=True)
print("\n4.4 Relative income index", rel_index.shape)
print(np.round(rel_index, 2))

# 4.5 Boolean masks
below_08_2024 = S[rel_index[:, -1] < 0.8]
lower_in_2024 = S[rel_index[:, -1] < rel_index[:, 0]]
print("\n4.5 Index strictly below 0.8 in 2024:", below_08_2024.tolist())
print("    Index lower in 2024 than 2016:   ", lower_in_2024.tolist())

4.1 Mean / median of the 16 state medians
  2016: mean = 5235.94, median = 4636.50
  2019: mean = 5915.12, median = 5275.00
  2020: mean = 5276.19, median = 4634.00
  2022: mean = 6212.69, median = 5552.00
  2024: mean = 6741.94, median = 6109.00

4.2 Growth in median income 2016-2024
  Johor 2016-2024: 36.45%
  Kedah 2016-2024: 28.44%
  Kelantan 2016-2024: 32.61%
  Melaka 2016-2024: 23.32%
  Negeri Sembilan 2016-2024: 22.10%
  Pahang 2016-2024: 25.03%
  Perak 2016-2024: 17.00%
  Perlis 2016-2024: 17.75%
  Pulau Pinang 2016-2024: 36.55%
  Sabah 2016-2024: 18.98%
  Sarawak 2016-2024: 32.21%
  Selangor 2016-2024: 48.46%
  Terengganu 2016-2024: 41.18%
  W.P. Kuala Lumpur 2016-2024: 19.06%
  W.P. Labuan 2016-2024: 24.54%
  W.P. Putrajaya 2016-2024: 30.14%
  Highest: Selangor 48.46%
  Lowest:  Perak 17.00%

4.3 Highest / lowest median income by year
  2016: highest = W.P. Kuala Lumpur, lowest = Kelantan
  2019: highest = W.P. Kuala Lumpur, lowest = Kelantan
  2020: highest = W.P. Putrajaya,

In [7]:
state_to_region = create_region_mapping(REGIONS, STATES)   # raises if a state has no region or two regions
S = np.array(STATES)

print("4.6 Unweighted mean of state median income, by region")
region_means = {}
for region in REGIONS:
    mask = np.array([state_to_region[s] == region for s in STATES])   # Boolean mask for this region
    region_means[region] = MEDIAN[mask].mean(axis=0)
    print(f"  {region}: {np.round(region_means[region], 2)}   states: {S[mask].tolist()}")

4.6 Unweighted mean of state median income, by region
  Northern: [4357.5  4840.25 4276.25 5027.75 5479.5 ]   states: ['Kedah', 'Perak', 'Perlis', 'Pulau Pinang']
  Central: [7288.   8436.75 7653.5  8874.75 9472.  ]   states: ['Negeri Sembilan', 'Selangor', 'W.P. Kuala Lumpur', 'W.P. Putrajaya']
  Southern: [5620.  6240.5 5568.5 6544.5 7301.5]   states: ['Johor', 'Melaka']
  East Coast: [3917.33 4516.   3944.33 4748.33 5228.33]   states: ['Kelantan', 'Pahang', 'Terengganu']
  East Malaysia: [4733.67 5168.33 4576.67 5486.33 5925.67]   states: ['Sabah', 'Sarawak', 'W.P. Labuan']


In [8]:
res = analyze_matrices(MEDIAN, MEAN, POVERTY, STATES, REGIONS)
flagged = res["flagged_matrix"]
periods = ["2019-2020", "2020-2022", "2022-2024"]

print("4.7 Income rose but absolute poverty did not fall")
print("shape:", flagged.shape, "dtype:", flagged.dtype)
for i, s in enumerate(STATES):
    for j, p in enumerate(periods):
        if flagged[i, j]:
            print(f"  {s}, {p}")

4.7 Income rose but absolute poverty did not fall
shape: (16, 3) dtype: bool
  W.P. Kuala Lumpur, 2020-2022
  W.P. Putrajaya, 2022-2024


In [9]:
profile = build_state_profile("Perlis", STATES, SURVEY_YEARS, MEDIAN, MEAN, POVERTY, POVERTY_YEARS, REGIONS)
print(json.dumps(profile, indent=2))      # proves it is JSON-serializable

try:
    build_state_profile("Atlantis", STATES, SURVEY_YEARS, MEDIAN, MEAN, POVERTY, POVERTY_YEARS, REGIONS)
except ValueError as e:
    print("Raised as expected:", e)

{
  "state": "Perlis",
  "region": "Northern",
  "survey_years": [
    2016,
    2019,
    2020,
    2022,
    2024
  ],
  "median_incomes": [
    4204.0,
    4594.0,
    4043.0,
    4713.0,
    4950.0
  ],
  "mean_incomes": [
    4998.0,
    5476.0,
    4971.0,
    5664.0,
    5996.0
  ],
  "growth_percentage": 17.74500475737393,
  "latest_relative_index": 0.8102799148796858,
  "latest_rank": 12,
  "region_latest_mean": 5479.5,
  "above_region_mean": false,
  "absolute_poverty": [
    3.9,
    9.9,
    4.0,
    3.6
  ],
  "poverty_years": [
    2019,
    2020,
    2022,
    2024
  ],
  "flagged_periods": []
}
Raised as expected: Unknown state: Atlantis


## Task 5 — Integration, testing, interpretation, and reproducibility

### 5.2 Unit and integration tests

The test suite is in `test_solution.py`. It covers valid/invalid records, duplicate keys, joins, numeric conversion, matrix dimensions and missing values, focus-state calculations, regional mappings, relative-income calculations, state profiles, and the supplied raw datasets. From the project folder run:

```bash
python -m unittest -v test_solution.py
```

The current test suite contains 26 tests. All 26 passed in the checked environment.

### 5.3 Comparison table and interpretation

The group focus states below are generated from the five matric numbers using `get_group_states()`. For a fair comparison, poverty is compared from 2019 to 2024 because the selected poverty series begins in 2019; it is not presented as a 2016–2019 change. Income growth is the percentage change in median household income from 2016 to 2024. Poverty change is in percentage points.


In [10]:
# Task 5.3: reproducible comparison table for the group's assigned focus states
matric_numbers = [313464, 309324, 310509, 308161, 303072]
focus_states = get_group_states(matric_numbers, income_records)
comparison_rows = []
for state in focus_states:
    i = STATES.index(state)
    income_growth = (MEDIAN[i, -1] - MEDIAN[i, 0]) / MEDIAN[i, 0] * 100
    poverty_change = POVERTY[i, -1] - POVERTY[i, 0]  # 2019 -> 2024, percentage points
    comparison_rows.append((state, MEDIAN[i, 0], MEDIAN[i, -1], income_growth,
                            POVERTY[i, 0], POVERTY[i, -1], poverty_change))

print(f"{'State':22s} {'Median 2016':>12s} {'Median 2024':>12s} {'Income growth':>14s} {'Poverty 2019':>13s} {'Poverty 2024':>13s} {'Poverty change (pp)':>19s}")
for row in comparison_rows:
    state, inc16, inc24, growth, pov19, pov24, pov_change = row
    print(f"{state:22s} {inc16:12.1f} {inc24:12.1f} {growth:13.2f}% {pov19:13.2f} {pov24:13.2f} {pov_change:19.2f}")


State                   Median 2016  Median 2024  Income growth  Poverty 2019  Poverty 2024 Poverty change (pp)
Johor                        5652.0       7712.0         36.45%          3.90          2.50               -1.40
Kedah                        3811.0       4895.0         28.44%          8.80          8.40               -0.40
Terengganu                   4694.0       6627.0         41.18%          6.10          5.80               -0.30
W.P. Kuala Lumpur            9073.0      10802.0         19.06%          0.20          1.00                0.80
Pulau Pinang                 5409.0       7386.0         36.55%          1.90          1.90                0.00


#### Interpretation

- All five focus states show higher median household income in 2024 than in 2016, but the size of the increase differs by state.
- A positive income-growth figure does not automatically mean poverty fell. Check the poverty-change column separately: a negative value indicates a lower absolute-poverty rate by 2024; zero indicates no net change; a positive value indicates an increase.
- These are descriptive comparisons, not proof that income growth caused a change in poverty. The poverty series has no 2016 value in the selected matrix, so a 2016–2019 poverty comparison is not supported by this analysis.
- The group should discuss differences between states rather than claiming that one metric alone captures household wellbeing.

### 5.4 Kernel and reproducibility check

1. Keep both saved datasets in `data/raw/` and the Python files in this same project folder.
2. Install dependencies with `python -m pip install -r requirements.txt`.
3. Run `python -m unittest -v test_solution.py`; all 26 tests should pass.
4. In Jupyter, choose **Kernel → Restart Kernel and Run All Cells**. Confirm the notebook runs from the first cell to the last without relying on variables left over from an earlier session.
5. The notebook's original analysis cells use local JSON copies; the fetch helper uses `data/raw/` as its cache. The supplied data files make the results repeatable without needing a live API request.
